In [ ]:
# 1. Install Libraries
!pip install -q transformers accelerate scikit-learn peft trl bitsandbytes

import os
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments
from google.colab import drive
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

# Fix seed for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [ ]:
#cell 2

from huggingface_hub import login
login()

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.2-1B"

# Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
model.eval()

# Punctuation ID Mapping
PUNCT_MAP = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("-", add_special_tokens=False)[-1]
}
EOS_ID = tokenizer.eos_token_id
print(f"Punctuation IDs: {PUNCT_MAP}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Punctuation IDs: {'COMMA': 11, 'PERIOD': 13, 'QMARK': 30}


In [ ]:
# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. User-provided paths and loading code
BASE_PATH = "/content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2"
VAL_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_validation.Y.txt")
TEST_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_test.Y.txt")
TRAIN_Y_PATH = os.path.join(BASE_PATH, "iwslt2017_en_train.Y.txt")

# Fine-tuning results save path
OUTPUT_DIR = "/content/drive/MyDrive/experiments/llama_finetune0122"

print(f"Train Data Path: {TRAIN_Y_PATH}")
print(f"Output Model Path: {OUTPUT_DIR}")

print(f"Loading file: {VAL_Y_PATH}")
with open(VAL_Y_PATH, "r", encoding="utf-8") as f:
    val_y_list = [line.strip() for line in f if line.strip()]

with open(TEST_Y_PATH, "r", encoding="utf-8") as f:
    test_y_list = [line.strip() for line in f if line.strip()]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train Data Path: /content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2/iwslt2017_en_train.Y.txt
Output Model Path: /content/drive/MyDrive/experiments/llama_finetune0122
Loading file: /content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2/iwslt2017_en_validation.Y.txt


In [ ]:
# 3. Data loading function (Addressing Dataset definition issue)
def load_full_dataset(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    # Here, we use the Dataset imported above.
    return Dataset.from_dict({"text": lines})

# 4. Execute
print(f"Loading data from: {TRAIN_Y_PATH}")
train_dataset = load_full_dataset(TRAIN_Y_PATH)
print(f"Dataset Loaded! Sample count: {len(train_dataset)}")

Loading data from: /content/drive/MyDrive/punct_data/iwslt2017/xy_label_v2/iwslt2017_en_train.Y.txt
Dataset Loaded! Sample count: 357117


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 # Use FP16 for memory efficiency
)

In [ ]:
# 5. LoRA Configuration
# ---------------------------------------------------------
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"] # Tune core modules
)

In [ ]:
# 6. [Key Change] SFTConfig Configuration
# Using SFTConfig instead of TrainingArguments, moving relevant arguments here.
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",    # [Moved] Dataset column name
    # max_seq_length=256,           # [Moved] Sentence length limit
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=50,
    fp16=True,
    save_strategy="epoch",
    report_to="none",
    # packing=False, # (Use if needed, default is False)
)

In [ ]:
# 7. Run Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=sft_config,              # Pass SFTConfig
    processing_class=tokenizer,   # [Modified] tokenizer -> changed to processing_class
    peft_config=peft_config,
)

print(">>> Start Full Fine-tuning...")
trainer.train()

# 8. Save
trainer.save_model(OUTPUT_DIR)
print(f">>> Model Saved to {OUTPUT_DIR}")

# Clean up memory
del model, trainer
torch.cuda.empty_cache()

Exception ignored in: <function _xla_gc_callback at 0x79dee629d440>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/lib/__init__.py", line 127, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


KeyboardInterrupt: 

In [ ]:
# ======================================================
# [Step 2] Load Fine-tuned Model (for Evaluation)
# ======================================================
from peft import PeftModel

# 1. Reload Base Model (Shell)
# Load with FP16, same as during training.
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, # "meta-llama/Llama-3.2-1B"
    device_map="auto",
    torch_dtype=torch.float16
)

# 2. Combine Trained LoRA Adapter (Core)
# Overwrite the Base Model with the training results saved in OUTPUT_DIR.
# Setting the variable name to 'model' allows existing experimental code to work as is.
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval() # Switch to evaluation mode (e.g., disable Dropout)

print(">>> Fine-tuned Llama-1B Loaded Successfully for Inference!")

# 3. Tokenizer Check (if needed)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

>>> Fine-tuned Llama-1B Loaded Successfully for Inference!


In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score
import re

from peft import PeftModel

# --- 1. Configuration ---
# Use the entire range to secure question mark (QMARK) samples
dataset_subset = val_y_list
CALIB_SIZE = len(dataset_subset)
K_VALUE = 4  # K=4 Tokens (Strict)

# Punctuation Token IDs
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("-", add_special_tokens=False)[-1]
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())

def parse_sentence_to_boundaries(text: str):
    tokens = text.strip().split()
    boundaries = []
    for tok in tokens:
        # Pattern: (content)(punctuation)(zero or more closing quotes/parentheses/brackets etc.)
        m = re.match(r'^(.*?)([,.?])(["\'\)\]\}]*)$', tok)

        if m:
            base, punct_char = m.group(1), m.group(2)

            if punct_char == ',':
                label = "COMMA"
            elif punct_char == '.':
                label = "PERIOD"
            elif punct_char == '?':
                label = "QMARK"

            word = base
        else:
            label = "O"
            word = tok

        # Remove quotation marks/parentheses from the word (minimum necessary)
        word = re.sub(r'[\"\'\(\)[\]{}]', '', word)

        # Do not treat tokens with only quotation marks (") as boundaries
        if word:
            boundaries.append({"word": word, "label": label})
            continue

        # Case 2: Tokens with only punctuation, no word (e.g., '."', ',"', '?")
        # This punctuation is assigned to the label of the preceding word
        if m and boundaries:
            # If the previous label is already punctuation, a policy is needed on whether to overwrite/retain
            # Usually, the last punctuation mark is stronger, so overwriting is recommended
            boundaries[-1]["label"] = label

        # Case 3: Ignore tokens with only quotation marks (") etc.
        # (Comes here if neither m nor word is present)

    return boundaries

# Helper Function: Calculating Joint Probability using Chunking
def get_joint_score_optimized(prefix_ids, target_ids):
    """
    prefix_ids: Context tokens so far
    target_ids: K future tokens (lookahead_tokens)
    """
    if not target_ids: return 0.0
    full_input = torch.tensor([prefix_ids + target_ids], device=device)
    with torch.no_grad():
        out = model(full_input)

    start_pos = len(prefix_ids) - 1
    # Extract Logits corresponding to the K future tokens
    rel_logits = out.logits[0, start_pos : start_pos + len(target_ids), :]
    log_probs = torch.log_softmax(rel_logits, dim=-1)
    t_ids_tensor = torch.tensor(target_ids, device=device)

    # Return joint probability of corresponding tokens
    return log_probs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()

In [ ]:
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score
import re

# --- 1. Configuration ---
# Use the entire range to secure question mark (QMARK) samples
dataset_subset = val_y_list
CALIB_SIZE = len(dataset_subset)
K_VALUE = 1  # K=4 Tokens (Strict)

# Punctuation Token IDs
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("-", add_special_tokens=False)[-1],
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())

# --- 2. Data Collection (Apply Strict Logic) ---
print(f"Step 1: Collecting scores for K={K_VALUE} (Strict K Tokens, Full Future)...")
raw_data = []

# Check EOS token (for Padding)
EOS_ID = tokenizer.eos_token_id

for sent_idx, y_true in enumerate(tqdm(dataset_subset)):
    # Parse text
    boundaries = parse_sentence_to_boundaries(y_true)

    if not boundaries: continue

    current_words = []
    for i, item in enumerate(boundaries):
        word = item['word']
        gold_label = item['label']
        current_words.append(word)

        # History Encoding
        history_text = " ".join(current_words)
        h_ids = tokenizer.encode(history_text, add_special_tokens=True)

        # [Safety Check] Check if tokenizer attaches EOS (Llama usually does not)
        if len(h_ids) > 0 and h_ids[-1] == EOS_ID:
             # If EOS is at the end, remove it (to prevent the model from misunderstanding that the sentence has ended)
            h_ids = h_ids[:-1]

        # Prepare Lookahead Token (Full Future & Strict Padding)
        # Tokenize all future words
        full_future_words = [b['word'] for b in boundaries[i+1:]] # All future words
        full_future_str = " ".join(full_future_words)

        if not full_future_str:
            # If there is no future text, fill only with EOS
            lookahead_tokens = [EOS_ID] * K_VALUE
        else:
            # Full Encoding
            full_future_ids = tokenizer.encode(" " + full_future_str, add_special_tokens=False)

            # Truncate K tokens from the beginning
            lookahead_tokens = full_future_ids[:K_VALUE]

            # [Strict Padding] If less than K tokens, fill with EOS
            if len(lookahead_tokens) < K_VALUE:
                padding_len = K_VALUE - len(lookahead_tokens)
                lookahead_tokens = lookahead_tokens + ([EOS_ID] * padding_len)

        # S0: No Punctuation Score
        s0 = get_joint_score_optimized(h_ids, lookahead_tokens)

        # Cost (Current Word Logits)
        with torch.no_grad():
            h_out = model(torch.tensor([h_ids], device=device))
        base_logprobs = torch.log_softmax(h_out.logits[0, -1, :], dim=-1)

        # Batch Candidates (Batching + Chunking)
        batch_input_ids = [h_ids + [pid] + lookahead_tokens for _, pid in PUNCT_LIST]
        batch_tensor = torch.tensor(batch_input_ids, device=device)
        with torch.no_grad():
            batch_out = model(batch_tensor)

        entry = {"gold": gold_label}
        for b_idx, (pname, pid) in enumerate(PUNCT_LIST):
            entry[f"{pname}_cost"] = base_logprobs[pid].item()

            # Extract Gain
            start_pos = len(h_ids)
            rel_logits = batch_out.logits[b_idx, start_pos : start_pos + len(lookahead_tokens), :]
            lprobs = torch.log_softmax(rel_logits, dim=-1)
            t_ids_tensor = torch.tensor(lookahead_tokens, device=device)
            p_joint_score = lprobs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()

            entry[f"{pname}_gain"] = p_joint_score - s0

        raw_data.append(entry)

        # Teacher Forcing Update
        if gold_label != "O":
            punct_char = "," if gold_label == "COMMA" else ("." if gold_label == "PERIOD" else "?")
            current_words[-1] = current_words[-1] + punct_char

df_scores = pd.DataFrame(raw_data)

# --- 3. Grid Search (Punctuation-Focused Metric) ---
print("\nStep 2: Searching for optimal parameters...")
alpha_range = np.arange(0.1, 0.95, 0.05)
threshold_range = np.arange(-3.0, 2.0, 0.25)

best_score = -1
best_params = {}
golds = df_scores["gold"].values

# Set labels excluding 'O' as evaluation targets
EVAL_LABELS = ["COMMA", "PERIOD", "QMARK"]

for alpha in alpha_range:
    for thresh in threshold_range:
        preds = []
        for _, row in df_scores.iterrows():
            best_p, max_s = "O", float("-inf")
            for pname in ["COMMA", "PERIOD", "QMARK"]:
                # Use Convex Combination as mentioned in the paper (experimental tuning)
                score = (alpha * row[f"{pname}_cost"]) + ((1 - alpha) * row[f"{pname}_gain"])
                if score > max_s:
                    max_s, best_p = score, pname
            if max_s <= thresh: best_p = "O"
            preds.append(best_p)

        # Set 'average="macro"' to reflect the importance of minority class (e.g., QMARK) performance.
        current_score = f1_score(golds, preds, average="macro", labels=EVAL_LABELS, zero_division=0)

        if current_score > best_score:
            best_score, best_params = current_score, {"alpha": alpha, "threshold": thresh}

print(f"\n=== K={K_VALUE} (Strict Token) Optimization Results ===")
print(f"Target Metric: Macro F1 (excluding 'O')")
print(f"Best Macro F1: {best_score:.4f}")
print(f"Optimal ALPHA: {best_params['alpha']:.2f}")
print(f"Optimal THRESHOLD: {best_params['threshold']:.2f}")

Step 1: Collecting scores for K=1 (Strict K Tokens, Full Future)...


  0%|          | 1/1501 [00:04<1:51:02,  4.44s/it]


KeyboardInterrupt: 

미래의 4 tokens lookahead 가정 시

최적의 alpha와 threshold 계산

alpha_range = np.arange(0.1, 0.95, 0.05)

threshold_range = np.arange(-3.0, 2.0, 0.25)


In [ ]:
import torch
import pandas as pd
import time
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import re
import numpy as np

# --- 1. Parameter Settings (K=4 Optimal Value) ---
ALPHA_K4 = 0.55       # Calibration Result
THRESHOLD_K4 = -0.25  # Calibration Result
K_VALUE = 1           # Strict K=4

# Output Order
LABELS_ORDER = ["O", "COMMA", "PERIOD", "QMARK"]

# Punctuation Token IDs
PUNCT_TOKEN_IDS = {
    "COMMA": tokenizer.encode(",", add_special_tokens=False)[-1],
    "PERIOD": tokenizer.encode(".", add_special_tokens=False)[-1],
    "QMARK": tokenizer.encode("-", add_special_tokens=False)[-1],
}
PUNCT_LIST = list(PUNCT_TOKEN_IDS.items())


# --- 2. Evaluation Loop (Apply 2-Pass Optimization) ---
print(f"Starting Final Evaluation (K={K_VALUE}, Strict Token Logic, 2-Pass Optimization)")
print(f"Params: Alpha={ALPHA_K4}, Threshold={THRESHOLD_K4}")
print("Metric: Words/second")

all_golds = []
all_preds = []
total_processed_words = 0
start_time = time.time()
EOS_ID = tokenizer.eos_token_id

for sent_idx, y_true in enumerate(tqdm(test_y_list)):
    boundaries = parse_sentence_to_boundaries(y_true)
    if not boundaries: continue

    current_words = []
    for i, item in enumerate(boundaries):
        word, gold_label = item['word'], item['label']
        current_words.append(word)
        total_processed_words += 1

        # History Encoding
        history_text = " ".join(current_words)
        h_ids = tokenizer.encode(history_text, add_special_tokens=True)
        if len(h_ids) > 0 and h_ids[-1] == EOS_ID: h_ids = h_ids[:-1]

        # [Strict K=4 Lookahead]
        full_future_words = [b['word'] for b in boundaries[i+1:]]
        full_future_str = " ".join(full_future_words)

        if not full_future_str:
            lookahead_tokens = [EOS_ID] * K_VALUE
        else:
            full_future_ids = tokenizer.encode(" " + full_future_str, add_special_tokens=False)
            lookahead_tokens = full_future_ids[:K_VALUE]
            if len(lookahead_tokens) < K_VALUE:
                lookahead_tokens = lookahead_tokens + ([EOS_ID] * (K_VALUE - len(lookahead_tokens)))

        # === 2-Pass Optimization Core ===
        # Pass 1: Simultaneously calculate (Cost) and (S0) by inputting History + Lookahead at once
        # -------------------------------------------------------------------------
        # Input structure: [h_1, ..., h_n, t_1, ..., t_k]
        full_input_ids = h_ids + lookahead_tokens
        full_input_tensor = torch.tensor([full_input_ids], device=device)

        with torch.no_grad():
            out_1 = model(full_input_tensor)

        # (1) Extract Cost: Logits at the position of the last history token (h_n)
        # Position of h_n is index `len(h_ids) - 1`
        # These Logits are "next token prediction" probabilities, so punctuation probabilities (Cost) are obtained from here.
        base_logits = out_1.logits[0, len(h_ids) - 1, :]
        base_logprobs = torch.log_softmax(base_logits, dim=-1)

        # (2) Extract S0: Probability from end of History to end of Lookahead (Joint Probability)
        # Logits range: `len(h_ids) - 1` (first Lookahead prediction) ~ `len(full_input) - 2` (last Lookahead prediction)
        start_pos = len(h_ids) - 1
        target_len = len(lookahead_tokens)

        # Extract Logits for the Lookahead part
        # out_1.logits[0, start_pos : start_pos + target_len] -> [K, Vocab]
        s0_logits = out_1.logits[0, start_pos : start_pos + target_len, :]
        s0_logprobs = torch.log_softmax(s0_logits, dim=-1)
        s0_target_ids = torch.tensor(lookahead_tokens, device=device)

        # Sum probabilities of correct tokens (Lookahead tokens)
        s0 = s0_logprobs.gather(1, s0_target_ids.unsqueeze(1)).squeeze(1).sum().item()

        # Pass 2: Batch Forward (Calculate Gain)
        # -------------------------------------------------------------------------
        batch_input_ids = [h_ids + [pid] + lookahead_tokens for _, pid in PUNCT_LIST]
        batch_tensor = torch.tensor(batch_input_ids, device=device)

        with torch.no_grad():
            batch_out = model(batch_tensor)

        best_p, max_s = "O", float('-inf')
        for b_idx, (pname, pid) in enumerate(PUNCT_LIST):
            # Use Cost obtained in Pass 1
            cost = base_logprobs[pid].item()

            # Extract Gain
            # Batch input structure: [h_1...h_n, PUNCT, t_1...t_k]
            # PUNCT position is len(h_ids). t_1 prediction comes from Logit at len(h_ids) position.
            start_pos_batch = len(h_ids)
            rel_logits = batch_out.logits[b_idx, start_pos_batch : start_pos_batch + len(lookahead_tokens), :]
            lprobs = torch.log_softmax(rel_logits, dim=-1)
            t_ids_tensor = torch.tensor(lookahead_tokens, device=device)
            p_joint_score = lprobs.gather(1, t_ids_tensor.unsqueeze(1)).squeeze(1).sum().item()

            gain = p_joint_score - s0

            score = (ALPHA_K4 * cost) + ((1 - ALPHA_K4) * gain)

            if score > THRESHOLD_K4 and score > max_s:
                max_s, best_p = score, pname

        all_golds.append(gold_label)
        all_preds.append(best_p)

        if best_p != "O":
            punct_char = "," if best_p == "COMMA" else ("." if best_p == "PERIOD" else "?")
            current_words[-1] = current_words[-1] + punct_char

# --- 3. Result Report ---
end_time = time.time()
elapsed = end_time - start_time
wps = total_processed_words / elapsed

print(f"\n[Final Results | K={K_VALUE} Strict 2-Pass Optimized]")
print(f"Inference Speed: {wps:.2f} words/s")
print(f"Total Processed Words: {total_processed_words}")
print(f"Total Execution Time: {elapsed:.2f}s")
print("-" * 60)
print(classification_report(all_golds, all_preds, labels=LABELS_ORDER, zero_division=0, digits=3))

print("\nConfusion Matrix")
cm = confusion_matrix(all_golds, all_preds, labels=LABELS_ORDER)
df_cm = pd.DataFrame(cm, index=[f"True_{l}" for l in LABELS_ORDER], columns=[f"Pred_{l}" for l in LABELS_ORDER])
print(df_cm)

Starting Final Evaluation (K=1, Strict Token Logic, 2-Pass Optimization)
Params: Alpha=0.55, Threshold=-0.25
Metric: Words/second


100%|██████████| 10799/10799 [2:42:31<00:00,  1.11it/s]



[Final Results | K=1 Strict 2-Pass Optimized]
Inference Speed: 18.90 words/s
Total Processed Words: 184278
Total Execution Time: 9751.38s
------------------------------------------------------------
              precision    recall  f1-score   support

           O      0.984     0.984     0.984    160196
       COMMA      0.799     0.808     0.804     13017
      PERIOD      0.989     0.986     0.987     10153
       QMARK      0.956     0.914     0.935       912

    accuracy                          0.971    184278
   macro avg      0.932     0.923     0.927    184278
weighted avg      0.971     0.971     0.971    184278


Confusion Matrix
             Pred_O  Pred_COMMA  Pred_PERIOD  Pred_QMARK
True_O       157580        2606           10           0
True_COMMA     2466       10522           29           0
True_PERIOD      76          33        10006          38
True_QMARK        1           2           75         834
